In [1]:
# Day 2 homework: validate and dispatch tool calls (my own build)
import json
import re
from copy import deepcopy

RECORDS = {
    'REF-1001': {'owner': 'demo-user-1', 'status': 'under_review'},
    'REF-1002': {'owner': 'demo-user-2', 'status': 'approved'},
}
ALLOWED_TOOLS = {'check_status'}
TRUSTED_USER = 'demo-user-1'

print('Records:', list(RECORDS))
print('Allowed tools:', ALLOWED_TOOLS)
print('Trusted user:', TRUSTED_USER)

Records: ['REF-1001', 'REF-1002']
Allowed tools: {'check_status'}
Trusted user: demo-user-1


In [2]:
def is_valid_call(call):
    # The call must be a dict with exactly two keys: name and arguments
    if not isinstance(call, dict):
        return False
    if set(call) != {'name', 'arguments'}:
        return False
    return True

print(is_valid_call({'name': 'check_status', 'arguments': {'reference': 'REF-1001'}}))
print(is_valid_call('not a tool object'))
print(is_valid_call({'name': 'check_status'}))

True
False
False


In [3]:
def are_valid_arguments(arguments):
    # check_status accepts exactly one argument: reference
    if not isinstance(arguments, dict) or set(arguments) != {'reference'}:
        return False
    ref = arguments['reference']
    # The reference must be text: REF- followed by exactly 4 digits
    return isinstance(ref, str) and re.fullmatch(r'REF-[0-9]{4}', ref) is not None

print(are_valid_arguments({'reference': 'REF-1001'}))
print(are_valid_arguments({'reference': 1001}))
print(are_valid_arguments({'reference': 'REF-1001', 'owner': 'demo-user-2'}))

True
False
False


In [4]:
def check_status(reference, user):
    # Read-only tool: it only reads RECORDS and never changes them
    record = RECORDS.get(reference)
    # Same answer for a missing record and someone else's record
    if record is None or record['owner'] != user:
        return {'code': 'unavailable'}
    return {'code': 'ok', 'status': record['status']}


def dispatch(call, trusted_user):
    # 1) Shape of the call
    if not is_valid_call(call):
        return {'code': 'invalid_call'}
    # 2) Allowlist: only approved tools
    if not isinstance(call['name'], str) or call['name'] not in ALLOWED_TOOLS:
        return {'code': 'tool_not_allowed'}
    # 3) Arguments
    if not are_valid_arguments(call['arguments']):
        return {'code': 'invalid_arguments'}
    # 4) Authorization: the user comes from the app, not from the call
    return check_status(call['arguments']['reference'], trusted_user)

print(dispatch({'name': 'check_status', 'arguments': {'reference': 'REF-1001'}}, TRUSTED_USER))
print(dispatch({'name': 'check_status', 'arguments': {'reference': 'REF-1002'}}, TRUSTED_USER))

{'code': 'ok', 'status': 'under_review'}
{'code': 'unavailable'}


In [5]:
before = deepcopy(RECORDS)

cases = [
    ('Owned reference',           {'name': 'check_status', 'arguments': {'reference': 'REF-1001'}}, {'code': 'ok', 'status': 'under_review'}),
    ("Another user's reference",  {'name': 'check_status', 'arguments': {'reference': 'REF-1002'}}, {'code': 'unavailable'}),
    ('Absent reference',          {'name': 'check_status', 'arguments': {'reference': 'REF-9999'}}, {'code': 'unavailable'}),
    ('Disallowed tool',           {'name': 'delete_record', 'arguments': {'reference': 'REF-1001'}}, {'code': 'tool_not_allowed'}),
    ('Unexpected owner argument', {'name': 'check_status', 'arguments': {'reference': 'REF-1001', 'owner': 'demo-user-2'}}, {'code': 'invalid_arguments'}),
    ('Numeric reference',         {'name': 'check_status', 'arguments': {'reference': 1001}}, {'code': 'invalid_arguments'}),
    ('Not a call object',         'not a tool object', {'code': 'invalid_call'}),
]

rows = []
for number, (label, call, expected) in enumerate(cases, 1):
    actual = dispatch(call, TRUSTED_USER)
    assert actual == expected, (number, label, actual, expected)
    rows.append({'case': number, 'label': label, 'result': actual, 'passed': actual == expected})

assert RECORDS == before, 'Records changed!'

report = {'mode': 'simulated_tool_protocol', 'passed': len(rows), 'total': len(cases),
          'records_unchanged': RECORDS == before, 'cases': rows}
print(json.dumps(report, indent=2))


{
  "mode": "simulated_tool_protocol",
  "passed": 7,
  "total": 7,
  "records_unchanged": true,
  "cases": [
    {
      "case": 1,
      "label": "Owned reference",
      "result": {
        "code": "ok",
        "status": "under_review"
      },
      "passed": true
    },
    {
      "case": 2,
      "label": "Another user's reference",
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 3,
      "label": "Absent reference",
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 4,
      "label": "Disallowed tool",
      "result": {
        "code": "tool_not_allowed"
      },
      "passed": true
    },
    {
      "case": 5,
      "label": "Unexpected owner argument",
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 6,
      "label": "Numeric reference",
      "result": {
        "code": "invalid_arguments"
      },
  

## Day 2 homework: results and explanation

**What it does.** `dispatch` treats a tool call as untrusted input and checks it in a fixed order: (1) the call has exactly the keys `name` and `arguments`; (2) the tool name is on the allowlist (`check_status` only); (3) the arguments are valid (only `reference`, text in the form `REF-` plus four digits); (4) authorization: the user comes from the app (`TRUSTED_USER`), never from the call, and the record must belong to that user.

**Result.** All 7 cases match the expected result (7/7) and `RECORDS` is unchanged afterwards, because the tool is read-only.

| # | Case | Caught by | Result |
|---:|---|---|---|
| 1 | Owned reference (`REF-1001`) | passes every check | `ok`, `under_review` |
| 2 | Another user's reference (`REF-1002`) | authorization | `unavailable` |
| 3 | Absent reference (`REF-9999`) | authorization | `unavailable` |
| 4 | Disallowed tool (`delete_record`) | allowlist | `tool_not_allowed` |
| 5 | Unexpected `owner` argument | argument validation | `invalid_arguments` |
| 6 | Numeric reference instead of text | argument validation | `invalid_arguments` |
| 7 | Text instead of a call object | call-shape check | `invalid_call` |

Cases 2 to 7 are correct refusals, not errors. A missing record and another user's record return the same `unavailable`, so the answer does not reveal whether a record exists.

**Validation vs authorization.** Validation asks whether the call is well formed; authorization asks whether this user may see this record. A valid call can still be refused.

**Limitation.** This is simulated tool calling: the calls are written by hand and no model chooses the tool, and the checks cover only these seven cases.